# LAB 3 — Ensaio do arco: uma notícia contra os dados do ISP

**Disciplina** Tomada de Decisão Baseada em Dados para Gestores de Segurança Pública (CAES-PMERJ / FGV)
**Aula** 3 — Estudo assíncrono no eClass (11 de novembro de 2026, horário livre)
**Duração do lab** cerca de 2 h, em grupo, depois das leituras
**Entrega** Post do grupo no fórum do eClass até **11/11/2026, 23:59**, com o texto gerado na Parte 6

> **Para quem é este lab:** gestores que decidem. Não é preciso programar. O grupo escreve respostas entre aspas onde houver `TODO` e roda as células prontas (Shift+Enter).

## Antes de começar

1. Importe este arquivo no Databricks Free Edition: **Workspace → (seu usuário) → Import → File → `lab-3.ipynb`**.
2. Conecte ao compute *Serverless* e rode as células na ordem, de cima para baixo.
3. Combinem um horário para o grupo trabalhar junto (videochamada): uma pessoa compartilha a tela e digita.
4. Tragam do lab 1 a CISP, o delito e os critérios do grupo. Os critérios podem ser ajustados aqui, desde que o grupo registre o porquê.

**O cenário.** A **Blue Blood** é uma força policial fictícia cuja área coincide com o estado do Rio de Janeiro: Comando de Área = RISP, batalhão = AISP, companhia = CISP. Organização, pessoas e números internos são fictícios; os dados de criminalidade são reais e públicos (ISP-RJ).

**Este lab é o ensaio geral da aula 4.** O grupo percorre as cinco etapas do arco sobre **uma** notícia real. Na aula 4, o mesmo caminho é feito com a base completa do ISP Microdados.

| Etapa do arco | Neste lab |
| --- | --- |
| 1. Pressão por decisão | A notícia e a decisão que ela pede (Parte 1) |
| 2. Critérios de decisão | Critérios, pesos e limiares, antes de ver os dados (Parte 2) |
| 3. Coleta e armazenamento | Série pública do ISPDados e teste de acesso ao ISP Microdados (Parte 3) |
| 4. Processamento e visualização | Trimestre contra o mesmo trimestre do ano anterior, em número e em taxa (Parte 4) |
| 5. Decisão com critérios e risco | Concordar ou contestar a notícia, com o risco de errar (Parte 5) |

## Objetivos de aprendizagem

Ao final deste lab, o grupo será capaz de:

1. **Transformar** uma notícia numa decisão de gestão com prazo e custo de errar.
2. **Fixar** critérios e limiares **antes** de olhar os dados.
3. **Obter** uma série oficial do ISP e registrar de onde ela veio.
4. **Comparar** períodos equivalentes, em número absoluto e em taxa por 100 mil habitantes.
5. **Concluir** se a notícia se sustenta, declarando o risco de estar errado.

## Parte 1 — A notícia e a decisão (20 min)

Escolham **uma** notícia sobre criminalidade no Rio de Janeiro que traga uma imagem negativa de alguma esfera da segurança pública (polícia, governo, um bairro, um tipo de crime). Procurem no Google News.

Boas candidatas têm:

- um **número** ou uma tendência afirmada ("disparou", "caiu 30%", "recorde");
- um **lugar** que dê para ligar a uma CISP, AISP, RISP ou município;
- um **período** identificável.

O Chefe de Comunicação Social traz as opções; o grupo escolhe.

In [0]:
# Parte 1 — a notícia escolhida pelo grupo. Escreva ENTRE ASPAS.

GRUPO = 0  # TODO: número do grupo, de 1 a 8 (sem aspas)

NOTICIA = {
    "titulo": "TODO",
    "veiculo": "TODO",
    "data": "TODO",                 # ex.: "2026-10-28"
    "url": "TODO",
    "afirmacao": "TODO",            # a frase com o número, copiada da notícia
    "esfera_atingida": "TODO",      # ex.: "polícia militar", "governo do estado", "bairro X"
}

VARIACAO_AFIRMADA_PCT = None  # TODO: variação que a notícia afirma, em %, ex. 40 (sem aspas); None se não houver número

DECISAO = {
    "decisao_em_jogo": "TODO",      # que decisão de gestão a notícia está pedindo?
    "quem_decide": "TODO",
    "prazo": "TODO",
    "custo_de_errar": "TODO",
}

---

## Parte 2 — Critérios, pesos e limiares, antes dos dados (20 min)

Reaproveitem os critérios do lab 1 ou ajustem para esta notícia. A célula vem com o exemplo da aula 1.

**Escala de Saaty:** 1 igual · 3 moderada · 5 forte · 7 muito forte · 9 extrema · 2, 4, 6, 8 intermediários · 1/3, 1/5... quando o segundo importa mais.

In [0]:
# Parte 2 — critérios e julgamentos par a par (exemplo da aula 1; troque pelos do grupo).

CRITERIOS = ["Evidência", "Custo operacional", "Legitimidade"]

JULGAMENTOS = {
    ("Evidência", "Custo operacional"): 3,
    ("Evidência", "Legitimidade"): 2,
    ("Legitimidade", "Custo operacional"): 2,
}

In [0]:
# Célula pronta: pesos (média geométrica) e razão de consistência (CR). Só execute.
import numpy as np
import pandas as pd

def mostrar(df):
    try:
        display(df)      # Databricks / Jupyter
    except NameError:
        print(df.to_string())

def pesos_ahp(criterios, julgamentos):
    n = len(criterios)
    idx = {c: i for i, c in enumerate(criterios)}
    A = np.ones((n, n))
    for (a, b), v in julgamentos.items():
        if a not in idx or b not in idx:
            raise ValueError(f"O par ({a}, {b}) usa um critério que não está em CRITERIOS.")
        A[idx[a], idx[b]] = v
        A[idx[b], idx[a]] = 1 / v
    faltando = [(criterios[i], criterios[j]) for i in range(n) for j in range(i + 1, n)
                if (criterios[i], criterios[j]) not in julgamentos and (criterios[j], criterios[i]) not in julgamentos]
    if faltando:
        raise ValueError(f"Faltam julgamentos para: {faltando}")
    media_geo = np.prod(A, axis=1) ** (1 / n)
    w = media_geo / media_geo.sum()
    lambda_max = (A @ w / w).mean()
    indice_aleatorio = {3: 0.58, 4: 0.90, 5: 1.12}  # Saaty
    cr = ((lambda_max - n) / (n - 1)) / indice_aleatorio[n] if n >= 3 else 0.0
    return A, w, cr

if not 2 <= len(CRITERIOS) <= 5:
    raise ValueError("Use de 2 a 5 critérios.")

MATRIZ, PESOS_VALORES, CR = pesos_ahp(CRITERIOS, JULGAMENTOS)
PESOS = dict(zip(CRITERIOS, PESOS_VALORES.round(3)))

tabela = pd.DataFrame(MATRIZ, index=CRITERIOS, columns=CRITERIOS).round(2)
tabela["Peso"] = [f"{p:.0%}" for p in PESOS_VALORES]
mostrar(tabela)
print(f"Razão de consistência (CR) = {CR:.3f}")
print("Julgamentos coerentes: pode usar os pesos." if CR < 0.10 else
      "CR acima de 0,10: os julgamentos se contradizem. Revejam o par mais discutido e rodem de novo.")

**O limiar da Evidência vira número.** Escrevam agora, antes de ver a série, a partir de que variação o grupo considera que **há alta real** e a partir de que diferença o grupo diz que a notícia **exagera**. Os outros critérios ficam em texto.

In [0]:
# Parte 2 — limiares escritos ANTES de ver os dados.

LIMIAR_ALTA_REAL_PCT = 15         # TODO: acima desta variação (%) contra o mesmo trimestre do ano anterior, há alta real
TOLERANCIA_NOTICIA_PP = 10        # TODO: diferença (pontos percentuais) aceita entre o número da notícia e o do ISP

INDICADORES_E_LIMIARES = {
    "Evidência": {"indicador": "Variação do delito no trimestre contra o mesmo trimestre do ano anterior, na CISP (ISP)",
                  "limiar": f"Alta real se passar de {LIMIAR_ALTA_REAL_PCT}%"},
    "Custo operacional": {"indicador": "TODO", "limiar": "TODO"},
    "Legitimidade": {"indicador": "TODO", "limiar": "TODO"},
}

---

## Parte 3 — Coleta: a série pública do ISP (20 min)

O ISPDados publica, de graça e sem cadastro, as séries mensais por delegacia (CISP) desde 2003. A célula abaixo baixa a série e a população estimada por CISP, e registra a data e a fonte.

- **Série por CISP:** `BaseDPEvolucaoMensalCisp.csv`
- **População por CISP:** `PopulacaoEvolucaoMensalCisp.csv` (a última estimativa publicada é de julho de 2022; usamos a mais recente disponível)

**Delitos disponíveis (nome da coluna):** `roubo_rua` (soma de transeunte, celular e coletivo), `roubo_transeunte`, `roubo_celular`, `roubo_em_coletivo`, `roubo_veiculo`, `roubo_carga`, `roubo_comercio`, `total_roubos`, `furto_transeunte`, `furto_celular`, `total_furtos`, `letalidade_violenta`, `hom_doloso`, `lesao_corp_dolosa`, `estupro`, entre outros.

> Se a rede falhar, o notebook usa um **plano B embutido**: dados reais do ISP para a CISP 12 e a RISP 1 (roubo de rua, de transeunte e de celular, out/2024 a set/2026), baixados em 24/10/2026.

In [0]:
# Parte 3 — o recorte do grupo (o mesmo do lab 1).

CISP = 12             # TODO: número da CISP do grupo (sem aspas)
DELITO = "roubo_rua"  # TODO: nome da coluna do delito, da lista acima

In [0]:
# Célula pronta: baixa as séries do ISPDados (ou usa o plano B) e registra a fonte. Só execute.
import io
import urllib.request
from datetime import datetime

URL_SERIE = "https://www.ispdados.rj.gov.br/Arquivos/BaseDPEvolucaoMensalCisp.csv"
URL_POP = "https://www.ispdados.rj.gov.br/Arquivos/PopulacaoEvolucaoMensalCisp.csv"

# Plano B: dados reais do ISP (CISP 12, RISP 1), baixados em 24/10/2026. fase 2 = dado preliminar.
_B_CISP12 = [  # ano, mes, fase, roubo_rua, roubo_transeunte, roubo_celular
    (2024, 10, 3, 26, 18, 8), (2024, 11, 3, 25, 17, 7), (2024, 12, 3, 30, 14, 13), (2025, 1, 3, 57, 36, 18),
    (2025, 2, 3, 17, 9, 7), (2025, 3, 3, 32, 21, 11), (2025, 4, 3, 20, 12, 7), (2025, 5, 3, 64, 24, 39),
    (2025, 6, 3, 20, 12, 6), (2025, 7, 3, 28, 16, 12), (2025, 8, 3, 34, 12, 19), (2025, 9, 3, 22, 11, 11),
    (2025, 10, 3, 22, 15, 4), (2025, 11, 3, 25, 13, 11), (2025, 12, 3, 24, 10, 12), (2026, 1, 3, 36, 19, 17),
    (2026, 2, 3, 32, 16, 15), (2026, 3, 3, 25, 16, 9), (2026, 4, 3, 25, 14, 9), (2026, 5, 3, 70, 35, 34),
    (2026, 6, 3, 26, 8, 18), (2026, 7, 2, 20, 10, 10), (2026, 8, 2, 21, 14, 5), (2026, 9, 2, 20, 10, 9)]
_B_RISP1 = [  # ano, mes, roubo_rua, roubo_transeunte, roubo_celular (soma das 26 CISPs da RISP 1)
    (2024, 10, 1799, 861, 663), (2024, 11, 1503, 749, 603), (2024, 12, 1821, 901, 754), (2025, 1, 2028, 1048, 738),
    (2025, 2, 1946, 972, 775), (2025, 3, 2491, 1265, 1044), (2025, 4, 1600, 783, 681), (2025, 5, 1645, 772, 772),
    (2025, 6, 1650, 737, 776), (2025, 7, 1669, 723, 863), (2025, 8, 1561, 632, 806), (2025, 9, 1574, 658, 814),
    (2025, 10, 1553, 668, 815), (2025, 11, 1412, 620, 721), (2025, 12, 1605, 703, 805), (2026, 1, 1717, 833, 791),
    (2026, 2, 1815, 916, 816), (2026, 3, 1691, 848, 748), (2026, 4, 1478, 727, 669), (2026, 5, 1515, 736, 681),
    (2026, 6, 1426, 635, 688), (2026, 7, 1498, 653, 669), (2026, 8, 1363, 634, 570), (2026, 9, 1275, 602, 565)]
_B_POP = {"cisp12": 98287.56, "risp1": 3034808}  # estimativa mais recente de cada CISP (até julho de 2022)

class EntradaInvalida(Exception):
    pass

def _ler_csv(url, **kw):
    with urllib.request.urlopen(url, timeout=60) as r:
        return pd.read_csv(io.BytesIO(r.read()), sep=";", encoding="latin1", **kw)

FONTE = {"data_hora": datetime.now().strftime("%Y-%m-%d %H:%M")}
try:
    serie_todas = _ler_csv(URL_SERIE)
    pop_todas = _ler_csv(URL_POP, decimal=",")
    if CISP not in set(serie_todas["cisp"]):
        raise EntradaInvalida(f"A CISP {CISP} não existe na base do ISP. Confira o número.")
    if DELITO not in serie_todas.columns:
        raise EntradaInvalida(f"O delito '{DELITO}' não é uma coluna da base. Confira a lista acima.")
    risp = int(serie_todas.loc[serie_todas["cisp"] == CISP, "risp"].iloc[-1])
    cisps_risp = serie_todas.loc[serie_todas["risp"] == risp, "cisp"].unique()
    pop_ult = pop_todas.sort_values(["ano", "mes"]).groupby("circ")["pop_circ"].last()
    SERIE = (serie_todas[serie_todas["cisp"] == CISP][["ano", "mes", "fase", DELITO]]
             .rename(columns={DELITO: "casos"}))
    SERIE_RISP = (serie_todas[serie_todas["risp"] == risp].groupby(["ano", "mes"])[DELITO].sum()
                  .reset_index().rename(columns={DELITO: "casos"}))
    POP_CISP = float(pop_ult[CISP])
    POP_RISP = float(pop_ult[pop_ult.index.isin(cisps_risp)].sum())
    FONTE.update(origem="ISPDados (download agora)", url_serie=URL_SERIE, url_pop=URL_POP, linhas_baixadas=len(serie_todas))
except EntradaInvalida:
    raise
except Exception as erro:
    print(f"Rede indisponível ({type(erro).__name__}). Usando o plano B embutido: CISP 12, RISP 1.")
    if CISP != 12 or DELITO not in ("roubo_rua", "roubo_transeunte", "roubo_celular"):
        print("Atenção: o plano B só tem a CISP 12 e três delitos. O exemplo segue com eles; refaça com rede.")
        CISP = 12
        DELITO = DELITO if DELITO in ("roubo_rua", "roubo_transeunte", "roubo_celular") else "roubo_rua"
    col = {"roubo_rua": 3, "roubo_transeunte": 4, "roubo_celular": 5}[DELITO]
    risp = 1
    SERIE = pd.DataFrame([(r[0], r[1], r[2], r[col]) for r in _B_CISP12], columns=["ano", "mes", "fase", "casos"])
    SERIE_RISP = pd.DataFrame([(r[0], r[1], r[col - 1]) for r in _B_RISP1], columns=["ano", "mes", "casos"])
    POP_CISP, POP_RISP = _B_POP["cisp12"], _B_POP["risp1"]
    FONTE.update(origem="Plano B embutido (ISPDados, baixado em 24/10/2026)", url_serie=URL_SERIE, url_pop=URL_POP,
                 linhas_baixadas=0)

SERIE = SERIE.sort_values(["ano", "mes"]).reset_index(drop=True)
SERIE_RISP = SERIE_RISP.sort_values(["ano", "mes"]).reset_index(drop=True)
FONTE.update(cisp=CISP, risp=risp, delito=DELITO,
             ultimo_mes=f"{int(SERIE['mes'].iloc[-1]):02d}/{int(SERIE['ano'].iloc[-1])}")
for k, v in FONTE.items():
    print(f"{k}: {v}")

### Atividade 3.2: teste de acesso ao ISP Microdados (10 min)

Na aula 4, **cada aluno** baixa a base do [ISP Microdados](https://ispconecta.rj.gov.br/microdados/) com as próprias credenciais. Hoje, cada integrante só **testa o acesso**: abre a ferramenta, vê o formulário e os filtros, e **não baixa a base final**.

> **Portaria ISP 127/2022:** dados fora dos sites públicos do ISP são fornecidos mediante Termo de Responsabilidade; é proibido comercializar ou repassar a base a terceiros; o uso fica restrito a estudos próprios; inconsistências devem ser comunicadas ao ISP. Por isso o professor não distribui a base e cada um baixa a sua.

A Corregedora e Governança de Dados confere que todos registraram o teste.

In [0]:
# Atividade 3.2 — um registro por integrante. Não baixe a base final hoje.

TESTE_MICRODADOS = {
    "Comandante-Geral": {"data": "TODO", "abriu_o_formulario": "TODO", "observacao": ""},
    "Chefe de Inteligência e Análise Criminal": {"data": "TODO", "abriu_o_formulario": "TODO", "observacao": ""},
    "Chefe de Comunicação Social": {"data": "TODO", "abriu_o_formulario": "TODO", "observacao": ""},
    "Corregedora e Governança de Dados": {"data": "TODO", "abriu_o_formulario": "TODO", "observacao": ""},
    "Comandante de Área Operacional": {"data": "TODO", "abriu_o_formulario": "TODO", "observacao": ""},
}
# "abriu_o_formulario": "sim" ou "não". Se "não", descreva o problema em "observacao".

---

## Parte 4 — Processamento: compare períodos equivalentes (20 min)

A célula abaixo calcula, para o delito e a CISP do grupo:

- o **último trimestre** disponível contra o **mesmo trimestre do ano anterior** (tira o efeito da época do ano);
- o último trimestre contra o **trimestre imediatamente anterior** (o que muitas manchetes comparam);
- a **taxa por 100 mil habitantes**, na CISP e na RISP inteira, para saber se a CISP está acima ou abaixo da sua região.

**Armadilhas que a célula aponta:** base pequena (poucos casos fazem qualquer variação parecer enorme) e dados preliminares do ISP (`fase 2`, ainda sujeitos a revisão).

In [0]:
# Célula pronta: trimestres comparáveis, taxa por 100 mil e gráfico. Só execute.
import matplotlib.pyplot as plt

def trimestre(df, fim):
    """Soma de 3 meses terminando na posição 'fim' (inclusive)."""
    return df["casos"].iloc[fim - 2: fim + 1].sum()

n = len(SERIE)
if n < 15:
    raise ValueError("A série precisa de pelo menos 15 meses para comparar com o ano anterior.")
fim = n - 1
atual, ano_anterior, tri_anterior = trimestre(SERIE, fim), trimestre(SERIE, fim - 12), trimestre(SERIE, fim - 3)

def var(a, b):
    return (a - b) / b * 100 if b else float("nan")

meses = SERIE.apply(lambda r: f"{int(r.mes):02d}/{int(r.ano)}", axis=1)
VAR_ANO_PCT = var(atual, ano_anterior)
VAR_TRI_PCT = var(atual, tri_anterior)
risp_atual, risp_ano = trimestre(SERIE_RISP, len(SERIE_RISP) - 1), trimestre(SERIE_RISP, len(SERIE_RISP) - 13)
TAXA_CISP = atual / POP_CISP * 1e5
TAXA_RISP = risp_atual / POP_RISP * 1e5

RESULTADO = pd.DataFrame({
    "Período": [f"{meses.iloc[fim - 2]} a {meses.iloc[fim]}", f"{meses.iloc[fim - 14]} a {meses.iloc[fim - 12]}",
                f"{meses.iloc[fim - 5]} a {meses.iloc[fim - 3]}"],
    "Comparação": ["Último trimestre", "Mesmo trimestre do ano anterior", "Trimestre anterior"],
    "Casos na CISP": [atual, ano_anterior, tri_anterior],
    "Taxa por 100 mil no trimestre (CISP)": [round(x / POP_CISP * 1e5, 1) for x in (atual, ano_anterior, tri_anterior)],
})
mostrar(RESULTADO)
print(f"Variação contra o mesmo trimestre do ano anterior: {VAR_ANO_PCT:+.0f}%")
print(f"Variação contra o trimestre anterior: {VAR_TRI_PCT:+.0f}%")
print(f"Taxa no último trimestre: CISP {TAXA_CISP:.1f} por 100 mil · RISP {risp} {TAXA_RISP:.1f} por 100 mil "
      f"(RISP variou {var(risp_atual, risp_ano):+.0f}% no mesmo período)")

AVISOS = []
if min(atual, ano_anterior) < 30:
    AVISOS.append(f"Base pequena: {ano_anterior} casos no trimestre de comparação. Variações percentuais oscilam muito.")
if (SERIE["fase"].iloc[fim - 2: fim + 1] < 3).any():
    AVISOS.append("O último trimestre tem dados preliminares do ISP (fase 2), sujeitos a revisão.")
AVISOS.append("População: estimativa mais recente do ISP por CISP (até julho de 2022); a taxa recente pode estar um pouco superestimada ou subestimada.")
for a in AVISOS:
    print("Aviso:", a)

ult = SERIE.tail(24).reset_index(drop=True)
rot = meses.tail(24).tolist()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
cores = ["#1d6fb8" if i >= len(ult) - 3 else ("#8a96a3" if len(ult) - 15 <= i < len(ult) - 12 else "#c9d3dd")
         for i in range(len(ult))]
ax1.bar(rot, ult["casos"], color=cores)
ax1.set_title(f"CISP {CISP} · {DELITO} · casos por mês")
ax1.set_ylabel("casos")
taxa_cisp = ult["casos"] / POP_CISP * 1e5
taxa_risp = SERIE_RISP.tail(24)["casos"].values / POP_RISP * 1e5
ax2.plot(rot, taxa_cisp, marker="o", color="#0b2a4a", label=f"CISP {CISP}")
ax2.plot(rot, taxa_risp, linestyle="--", color="#8a96a3", label=f"RISP {risp}")
ax2.set_title("Taxa por 100 mil habitantes, por mês")
ax2.legend()
for ax in (ax1, ax2):
    ax.tick_params(axis="x", rotation=90, labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Azul: último trimestre · cinza escuro: mesmo trimestre do ano anterior", fontsize=10)
plt.tight_layout()
plt.show()

---

## Parte 5 — Decisão com critérios e risco (20 min)

A célula abaixo aplica os limiares que o grupo escreveu **antes** de ver os dados e sugere uma leitura. A posição final é do grupo, que pode discordar da sugestão, desde que explique por quê.

In [0]:
# Célula pronta: aplica os limiares da Parte 2. Só execute.

if VARIACAO_AFIRMADA_PCT is None:
    LEITURA = "Sem número na notícia: compare a tendência afirmada com a variação do ISP."
elif abs(VARIACAO_AFIRMADA_PCT - VAR_ANO_PCT) <= TOLERANCIA_NOTICIA_PP:
    LEITURA = "O ISP CONFIRMA a ordem de grandeza da notícia."
elif abs(VARIACAO_AFIRMADA_PCT - VAR_TRI_PCT) <= TOLERANCIA_NOTICIA_PP:
    LEITURA = ("O número da notícia bate com a comparação contra o trimestre ANTERIOR, que mistura época do ano. "
               "Contra o mesmo trimestre do ano anterior, o quadro é outro.")
else:
    LEITURA = "O ISP CONTESTA a notícia: a variação oficial é bem diferente da afirmada."

ALTA_REAL = VAR_ANO_PCT > LIMIAR_ALTA_REAL_PCT
afirmado = "sem número" if VARIACAO_AFIRMADA_PCT is None else f"{VARIACAO_AFIRMADA_PCT}%"
print(f"Notícia afirma: {afirmado} · ISP, mesmo trimestre do ano anterior: {VAR_ANO_PCT:+.0f}% · "
      f"trimestre anterior: {VAR_TRI_PCT:+.0f}%")
print("Leitura sugerida:", LEITURA)
print(f"Pelo limiar do grupo ({LIMIAR_ALTA_REAL_PCT}%), há alta real na CISP? {'SIM' if ALTA_REAL else 'NÃO'}")

In [0]:
# Parte 5 — a posição do grupo.

POSICAO = "TODO"            # "concorda", "contesta" ou "inconclusivo"
DECISAO_FINAL = "TODO"      # o que o comando faz agora (uma ou duas frases)
RISCO_DE_ERRAR = "TODO"     # o que pode estar errado na conclusão e o que acontece se estiver
INDICADOR_DE_ACOMPANHAMENTO = "TODO"  # que número o grupo acompanha nos próximos meses

---

## Parte 6 — Rodada 3 da Operação Manchete e o post no fórum (10 min)

**Rodada 3 — O atalho.** Um analista quer mandar a base do ISP Microdados por e-mail aos batalhões e a um consultor externo "para ganhar tempo".

| Porta | Escolha |
| --- | --- |
| A | Autorizar o envio: "é dado público" |
| B | Cada unidade baixa a base com a própria credencial e registra data, filtros e o Termo de Responsabilidade |
| C | Batalhões usam só as séries públicas do ISPDados; microdados ficam na Inteligência |
| D | Proibir o uso de microdados no comando |
| E | Publicar a base tratada no site da Blue Blood |

O grupo escolhe uma porta e justifica em uma frase. As consequências e os pontos são revelados pelo professor na abertura da aula 4.

In [0]:
# Parte 6 — Rodada 3.

PORTA_RODADA_3 = "TODO"          # "A", "B", "C", "D" ou "E"
JUSTIFICATIVA_RODADA_3 = "TODO"  # uma frase

In [0]:
# Célula pronta: confere o que falta e gera o texto do fórum. Só execute.

def preenchido(v):
    if v is None:
        return False
    if isinstance(v, str):
        return v.strip() not in ("", "TODO")
    if isinstance(v, dict):
        return all(preenchido(x) for k, x in v.items() if k != "observacao")
    return True

itens = {
    "Parte 1 — número do grupo (1 a 8)": isinstance(GRUPO, int) and 1 <= GRUPO <= 8,
    "Parte 1 — notícia completa": preenchido(NOTICIA),
    "Parte 1 — decisão nomeada": preenchido(DECISAO),
    "Parte 2 — julgamentos coerentes (CR < 0,10)": CR < 0.10,
    "Parte 2 — indicador e limiar para cada critério": all(
        c in INDICADORES_E_LIMIARES and preenchido(INDICADORES_E_LIMIARES[c]) for c in CRITERIOS),
    "Parte 3 — série do ISP carregada da rede (não do plano B)": FONTE["origem"].startswith("ISPDados"),
    "Parte 3 — teste do ISP Microdados registrado pelos 5": preenchido(TESTE_MICRODADOS),
    "Parte 5 — posição do grupo": POSICAO in ("concorda", "contesta", "inconclusivo"),
    "Parte 5 — decisão, risco e acompanhamento": all(preenchido(x) for x in (DECISAO_FINAL, RISCO_DE_ERRAR, INDICADOR_DE_ACOMPANHAMENTO)),
    "Parte 6 — porta e justificativa da Rodada 3": PORTA_RODADA_3 in ("A", "B", "C", "D", "E") and preenchido(JUSTIFICATIVA_RODADA_3),
}
for item, ok in itens.items():
    print(("OK    " if ok else "FALTA ") + item)

if all(itens.values()):
    criterios_txt = "; ".join(f"{c} ({PESOS[c]:.0%}): {INDICADORES_E_LIMIARES[c]['limiar']}" for c in CRITERIOS)
    print(f"""
==== copie a partir daqui para o fórum ====
Grupo {GRUPO} — Ensaio do arco (lab 3)

1. Pressão: "{NOTICIA['titulo']}" ({NOTICIA['veiculo']}, {NOTICIA['data']}) {NOTICIA['url']}
   Afirmação: {NOTICIA['afirmacao']}
   Decisão em jogo: {DECISAO['decisao_em_jogo']} · decide: {DECISAO['quem_decide']} · prazo: {DECISAO['prazo']}
2. Critérios (CR = {CR:.2f}): {criterios_txt}
3. Dados: ISP, CISP {CISP}, {DELITO}, até {FONTE['ultimo_mes']} (baixado em {FONTE['data_hora']}).
   Teste de acesso ao ISP Microdados registrado pelos 5 integrantes.
4. Resultado: {VAR_ANO_PCT:+.0f}% contra o mesmo trimestre do ano anterior; {VAR_TRI_PCT:+.0f}% contra o trimestre anterior;
   taxa de {TAXA_CISP:.1f} por 100 mil na CISP contra {TAXA_RISP:.1f} na RISP {risp}.
5. Posição: {POSICAO}. Decisão: {DECISAO_FINAL}
   Risco de errar: {RISCO_DE_ERRAR}
   Acompanhamento: {INDICADOR_DE_ACOMPANHAMENTO}
Rodada 3: porta {PORTA_RODADA_3} — {JUSTIFICATIVA_RODADA_3}
==== fim ====""")

## Material de apoio

- **Roteiro da aula 3** (slides no eClass): leituras, os cinco passos do ensaio e o modelo do post.
- [ISPDados — dados abertos](https://www.ispdados.rj.gov.br/estatistica.html): séries mensais por CISP e população estimada.
- [ISP Microdados](https://ispconecta.rj.gov.br/microdados/) e [Portaria ISP nº 127/2022](http://arquivo.proderj.rj.gov.br/isp_imagens/Uploads/PortariaISP127_Fornecimento_Microdados.pdf).
- INSTITUTO SOU DA PAZ. [Segurança Pública em Dados: Guia Prático para Jornalistas](https://lp.soudapaz.org/guiajornalistas) — como checar um número de segurança.

**Na aula 4 (12/11, presencial):** o arco completo com a base do ISP Microdados, NLP sobre as narrativas, dashboard, decisão e proposta de política pública; apresentação dos grupos à tarde.